# Classes

Bundling data with the functions that work on it, and describing data precisely.

# Part 1: Classes

## State that lives on between calls

From last Friday: a function that returns a function, and a name the inner function keeps.

In [ ]:
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment


tick = make_counter()
tick()
tick()
print(tick())

This works, but `count` is hidden: nothing outside can read it, and a second operation (say, `reset`) would need a second inner function that shares it. It gets worse with more state. In the exercise 4 bonus, memory, position and relative base belong together, and every function needs all three.

A **class** puts the state and the functions that work on it in one place.

## A class

In [ ]:
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1
        return self.count

- `class Counter:` makes a new type, like `int` or `list`, called `Counter`.
- `Counter()` makes a new object of that type, an **instance**. Python then calls `__init__` on it.
- `self` is the object the method works on. `self.count = 0` sticks a name, an **attribute**, onto that object.
- A function defined inside a class is a **method**. You call it with a dot: `a.increment()`.

**Predict:**

In [ ]:
a = Counter()
b = Counter()
a.increment()
a.increment()
print(a.increment())

**Predict:**

In [ ]:
print(b.count)

In [ ]:
print(type(a))

Printing the object itself:

In [ ]:
print(a)

Every instance has its own attributes: `a.count` is 3, `b.count` is still 0. `print(a)` shows the type and where the object lives in memory; we'll make that nicer below.

## Names

How code should be laid out and named is written down in **PEP 8**, Python's style guide. Most of what ruff checks in our CI comes from it, and `ruff format` takes care of the spacing. Names are up to you:

| Kind | Style | Examples |
|---|---|---|
| classes | `CapWords` | `Counter`, `MobilePhone`, `ValueError` |
| functions, methods, variables, attributes | `snake_case` | `make_counter`, `self.count`, `load_program` |
| constants | `UPPER_CASE` | `PROGRAM_FILE`, `MAX_OF_TWO` |
| internal (see below) | leading `_` | `self._balance` |

So `Counter()` makes an object and `count_tiles()` calls a function: you can tell them apart before you know what they do.

## What `self` is

**Predict:**

In [ ]:
Counter.increment(b)
print(b.count)

`b.increment()` is short for `Counter.increment(b)`: Python looks the method up in the class and passes the object in front of the dot as the **first argument**. `self` is just the usual name for that parameter.

In [ ]:
print(type(Counter.increment))

In [ ]:
Counter.increment

In [ ]:
print(type(b.increment))

In [ ]:
print(b.increment)

**Predict:**

In [ ]:
step = b.increment
step()
step()
print(b.count)

Read from the class, `Counter.increment` is a plain function. Read from an object, `b.increment` is a **bound method**: the function and the object packed together. Calling it passes `b` as `self`, even after you've stuck another name on it (last Friday: a function without brackets is a value, `loud = shout`).

**Predict:**

In [ ]:
class Greeter:
    def hello():
        return "hello"


Greeter().hello()

**Predict:**

In [ ]:
class Forgetful:
    def __init__(self):
        count = 0

    def increment(self):
        self.count += 1
        return self.count


f = Forgetful()
f.increment()

Two common mistakes:

- `def hello():` without `self`: the call still passes the object, so `hello` gets one argument it has no parameter for (`TypeError: ... takes 0 positional arguments but 1 was given`).
- `count = 0` without `self.`: that's a local variable of `__init__`, gone when `__init__` returns. The object never got a `count` (`AttributeError`).

## `__init__` with arguments

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money")
        self.balance -= amount

**Predict:**

In [ ]:
ada = Account("Ada", 100)
ada.deposit(50)
ada.withdraw(30)
print(ada.balance)

**Predict:**

In [ ]:
ada.withdraw(500)

In [ ]:
print(ada.balance)

The methods guard the state: as long as everyone uses `deposit` and `withdraw`, the balance never goes below zero.

**Predict:**

In [ ]:
ada.balance = -1000
print(ada.balance)

Nothing stops you. Python has no truly private attributes; it trusts you. The convention: a name starting with one underscore, such as `self._balance`, means "internal, don't use it from outside the class". Nothing enforces it, but code that uses someone else's `_name` may break with their next version, and that's on you.

Two underscores go one step further:

In [ ]:
class Safe:
    def __init__(self, code):
        self.__code = code

    def opens_with(self, guess):
        return guess == self.__code


safe = Safe(1234)
print(safe.opens_with(1234))

In [ ]:
print(safe.__code)

In [ ]:
print(vars(safe))

In [ ]:
print(safe._Safe__code)

Inside the class, Python renames `__code` to `_Safe__code` (**name mangling**); `vars` shows an object's attributes as a dict. So `safe.__code` from outside finds nothing, but the value is still there for anyone who knows the pattern. It's not about security: it keeps a subclass that happens to use the same name `__code` from overwriting this one. One underscore is the usual choice.

## Passing objects around

**Predict:**

In [ ]:
bob = Account("Bob")
other = bob
other.deposit(10)
print(bob.balance)

**Predict:**

In [ ]:
def pay_bonus(account):
    account.deposit(100)


pay_bonus(bob)
print(bob.balance)

Two accounts with the same owner and balance:

In [ ]:
cy = Account("Cy", 5)
cy_again = Account("Cy", 5)
print(cy == cy_again)

Objects behave like lists and dicts in week 3: `other = bob` sticks a second label on the **same** object, and a function gets the object itself, not a copy. For your own classes, `==` means `is` (the same object) unless you say otherwise. That's next.

## Methods Python calls for you

Python calls `__init__` when you create an object. It calls other double-underscore methods ("dunder methods") too, if a class has them:

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)

**Predict:**

In [ ]:
print(Point(1, 2))

In [ ]:
print(Point(1, 2) == Point(1, 2))

In [ ]:
print(Point(1, 2) + Point(10, 20))

In [ ]:
print([Point(0, 0), Point(1, 1)])

In [ ]:
print(Point(1, 2) in [Point(0, 0), Point(1, 2)])

`in` on a list compares with `==`, so it uses `__eq__` too.

A `Point` as a dict key:

In [ ]:
places = {Point(1, 2): "home"}

A dict key needs a **hash**, a number computed from the value that must never change while the key is in the dict (week 3: that's why a list can't be a key). Once a class defines `__eq__`, Python stops giving its objects a hash, because the default one would disagree with the new `==`. A class can define `__hash__` as well (below); for positions as keys, a tuple or a frozen dataclass (Part 2) is simpler.

`__repr__` is meant for programmers: unambiguous, ideally looking like the code that makes the object. `__str__` is for display. `print` and f-strings use `__str__` if there is one, otherwise `__repr__`; a list shows its elements with `__repr__`.

In [ ]:
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __repr__(self):
        return f"Money({self.cents})"

    def __str__(self):
        return f"{self.cents / 100:.2f} €"

**Predict:**

In [ ]:
price = Money(250)
print(price)

In [ ]:
print([price])

## More methods Python calls for you

A tour of the common ones, by what they make work.

### Comparing and sorting

In [ ]:
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor

    def __repr__(self):
        return f"Version({self.major}, {self.minor})"

    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):
        return (self.major, self.minor) < (other.major, other.minor)

**Predict:**

In [ ]:
print(sorted(["1.10", "1.2", "0.9"]))

In [ ]:
print(sorted([Version(1, 10), Version(1, 2), Version(0, 9)]))

`Version` has no `__gt__` and no `__ne__`:

In [ ]:
print(Version(1, 2) > Version(1, 0))

In [ ]:
print(Version(1, 2) != Version(1, 3))

In [ ]:
print(Version(1, 2) <= Version(1, 2))

- `sorted`, `min` and `max` only need `<`, so `__lt__` is enough for them.
- `a > b`: if `a` has no `__gt__`, Python tries the mirror image, `b < a`.
- `!=` is `not ==` unless you say otherwise.
- `<=` is **not** `<` or `==`: without `__le__` (or `__ge__` on the other side), it's a `TypeError`. `functools.total_ordering`, or a dataclass with `order=True` (Part 2), fill in the rest.

### Arithmetic

In [ ]:
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __repr__(self):
        return f"Money({self.cents})"

    def __str__(self):
        return f"{self.cents / 100:.2f} €"

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.cents + other.cents)

    def __mul__(self, factor):
        return Money(self.cents * factor)

**Predict:**

In [ ]:
price = Money(250)
print(price + Money(100))

In [ ]:
print(price * 3)

In [ ]:
print(3 * price)

`price * 3` calls `price.__mul__(3)`. `3 * price` calls `int`'s `__mul__` first, which doesn't know `Money` and returns `NotImplemented`. Python then tries the mirror-image method on the right-hand side, `price.__rmul__(3)`, and `Money` doesn't have one. (`NotImplemented` is exactly what `__add__` returns for anything that isn't `Money`, so that `price + 5` gives a clear `TypeError` instead of a crash inside the method.)

In [ ]:
Money.__rmul__ = Money.__mul__
print(3 * price)

(Assigning a method to a class after the fact works, but write it in the class body normally: 

`def __rmul__(self, factor):
    return self * factor`.)

**Predict:**

In [ ]:
print(sum([Money(100), Money(250)]))

In [ ]:
print(sum([Money(100), Money(250)], Money(0)))

`sum` starts from `0`, and `0 + Money(100)` fails. Give it a start value of the right type.

**Predict:**

In [ ]:
wallet = price
price += Money(100)
print(price)

In [ ]:
print(wallet)

Without `__iadd__`, `price += x` means `price = price + x`: a **new** object for `price`, while `wallet` keeps the old one. For lists, `+=` changes the list in place, and `wallet` would see the change (week 3). Classes for values, like `Money`, should behave like `int`: never change in place.

### True or false

In [ ]:
print(bool(Money(0)))

Every object is true unless its class says otherwise, with `__bool__`, or with `__len__` returning 0. Next up: a class with `__len__`.

### Containers

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs.copy()

    def __len__(self):
        return len(self.songs)

    def __getitem__(self, index):
        return self.songs[index]

    def __contains__(self, song):
        return song in self.songs

    def __iter__(self):
        return iter(self.songs)


playlist = Playlist(["intro", "verse", "outro"])

**Predict:**

In [ ]:
print(len(playlist))

In [ ]:
print(playlist[-1])

In [ ]:
print(playlist[0:2])

In [ ]:
print("verse" in playlist)

In [ ]:
for song in playlist:
    print(song.upper())

In [ ]:
print(bool(Playlist([])))

`len`, `[...]`, `in` and `for` each call one method. `__getitem__` gets whatever is in the brackets: an index, a slice (and the list's slice is a plain list, not a `Playlist`), or a key for something dict-like. `__iter__` returns what the `for` loop walks through; `iter(self.songs)` hands on the list's. `__setitem__` and `__delitem__` would make `playlist[0] = ...` and `del playlist[0]` work.

### Calling an object

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return self.factor * x


triple = Multiplier(3)

**Predict:**

In [ ]:
print(triple(5))

In [ ]:
print([triple(n) for n in range(4)])

With `__call__`, an object can be used like a function. Compare last Friday's `make_adder`: a closure keeps its state hidden; here it's an attribute, `triple.factor`, that you can read.

### Hashing

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


home = Point(1, 2)
places = {home: "home"}

**Predict:**

In [ ]:
print(places[Point(1, 2)])

Now we move `home`:

In [ ]:
home.x = 5
print(Point(1, 2) in places)

In [ ]:
print(home in places)

In [ ]:
places

A dict finds a key by its hash first, and only then compares with `==`. Two rules: objects that are equal must have the same hash (hence the tuple of the same attributes as in `__eq__`), and a key's hash must never change. `home` was filed under the hash of `(1, 2)`: looking for `Point(1, 2)` finds it, but it isn't equal any more; looking for `home` now looks under `(5, 2)`, where nothing was filed. Only give a class `__hash__` if its objects don't change; that's what frozen dataclasses (Part 2) do for you.

### `with`

In [ ]:
import time


class Timer:
    def __enter__(self):
        print("start")
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.seconds = time.perf_counter() - self.start
        print("stop")

**Predict:** in which order are the lines printed?

In [ ]:
with Timer() as timer:
    print("working")
print(timer.seconds < 1)

**Predict:** does `stop` get printed?

In [ ]:
with Timer():
    print("working")
    int("oops")
    print("done")

`with` calls `__enter__` at the start (what it returns goes to the name after `as`) and `__exit__` at the end, **also when an error leaves the block**. `__exit__` gets the error's details (all `None` if there was none) and lets it carry on. That's how `with open(...)` (week 4) closes the file whatever happens.

### Overview

| Method | Called by |
|---|---|
| `__init__(self, ...)` | `Point(1, 2)` |
| `__repr__(self)`, `__str__(self)` | `repr(p)`, `str(p)`, `print(p)`, f-strings; a list shows its elements with `__repr__` |
| `__format__(self, spec)` | `f"{p:spec}"`, `format(p, spec)` |
| `__eq__`, `__ne__` | `==`, `!=` (`!=` is `not ==` by default) |
| `__lt__`, `__le__`, `__gt__`, `__ge__` | `<`, `<=`, `>`, `>=`; `sorted`, `min`, `max` need only `<` |
| `__hash__(self)` | `hash(p)`, dict keys, sets; equal objects need equal hashes |
| `__bool__(self)` | `if p:`, `bool(p)`; without it `__len__`, without both always true |
| `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__pow__` | `+`, `-`, `*`, `/`, `//`, `%`, `**` |
| `__radd__`, `__rsub__`, `__rmul__`, ... | the same, with your object on the right, when the left one returns `NotImplemented` |
| `__iadd__`, `__isub__`, ... | `+=`, `-=`, ...; without them, `a += b` is `a = a + b` |
| `__neg__`, `__abs__` | `-p`, `abs(p)` |
| `__int__`, `__float__` | `int(p)`, `float(p)` |
| `__len__(self)` | `len(p)` |
| `__getitem__`, `__setitem__`, `__delitem__` | `p[key]`, `p[key] = value`, `del p[key]` |
| `__contains__(self, item)` | `item in p` |
| `__iter__(self)` | `for x in p`, `list(p)`, `sorted(p)`, unpacking |
| `__call__(self, ...)` | `p(...)` |
| `__enter__`, `__exit__` | `with p:` |
| `__getattr__(self, name)` | `p.name`, when there's no such attribute |

You define them; you rarely call them yourself. Rarer ones exist (`__new__`, `__del__`, `__getattribute__`, `__set_name__`, ...); look them up when you meet them.

## Attributes on the class

In [ ]:
class Dog:
    species = "dog"

    def __init__(self, name):
        self.name = name

**Predict:**

In [ ]:
rex = Dog("Rex")
print(rex.species)

**Predict:**

In [ ]:
rex.species = "wolf"
print(rex.species)

In [ ]:
print(Dog("Fido").species)

`species` belongs to the class, and all instances share it. Looking up `rex.species` checks the object first, then its class. **Assigning** `rex.species = ...` creates an attribute on `rex` itself, which hides the class attribute for `rex` only.

**Predict:**

In [ ]:
class Pet:
    tricks = []

    def __init__(self, name):
        self.name = name

    def learn(self, trick):
        self.tricks.append(trick)


rex = Pet("Rex")
fido = Pet("Fido")
rex.learn("sit")
print(fido.tricks)

`self.tricks.append(...)` doesn't assign to `self.tricks`, so it finds the one list on the class and changes it, for every pet. The same trap as a mutable default argument in week 3. Every object that needs its own list makes it in `__init__`: `self.tricks = []`. Class attributes are fine for constants that are never changed.

## Rectangles

**Predict:**

In [ ]:
class StoredRectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.area = width * height


r = StoredRectangle(2, 3)
r.width = 10
print(r.area)

`area` was computed once, in `__init__`, and nobody updated it. `@property` (a decorator, as on Wednesday) turns a method into an attribute that's computed every time you read it:

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

**Predict:**

In [ ]:
r = Rectangle(2, 3)
r.width = 10
print(r.area)

No brackets: `r.area`, not `r.area()`. A property can also check what's assigned to it, with a **setter**:

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"{value} °C is below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

**Predict:**

In [ ]:
t = Temperature(20)
print(t.fahrenheit)

In [ ]:
t.celsius = -300

In [ ]:
Temperature(-300)

`self.celsius = celsius` in `__init__` goes through the setter as well, so a `Temperature` can't even be created below absolute zero. The value itself lives in `self._celsius`.

## Getters and setters

In Java or C#, and in some Python code, you'll see attributes behind methods:

In [ ]:
class GetterAccount:
    def __init__(self, balance):
        self._balance = balance

    def get_balance(self):
        return self._balance

    def set_balance(self, value):
        if value < 0:
            raise ValueError("balance can't be negative")
        self._balance = value


account = GetterAccount(100)
account.set_balance(account.get_balance() + 50)
print(account.get_balance())

In those languages that's the safe choice: if you start with a plain attribute and later need a check, every line that uses it has to change. In Python, start with a plain attribute. When you need a check, turn it into a property, and the code that uses it stays the same.

Remember `ada.balance = -1000`? Here's `Account` again, with `balance` as a property.

**Predict:**

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("balance can't be negative")
        self._balance = value


ada = Account("Ada", 100)
ada.balance += 50
print(ada.balance)

**Predict:**

In [ ]:
ada.balance = -1000

`ada.balance += 50` reads through the getter and writes through the setter. Code written for the old `Account` still works, and the balance can't go negative any more, not even by assigning to it directly.

## A second way to make an object

A **class method** gets the class as its first argument (`cls`) instead of an object. Its most common use is another way to create objects:

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    @classmethod
    def from_text(cls, text):
        x, y = text.split(",")
        return cls(int(x), int(y))

    @staticmethod
    def origin_name():
        return "the origin"

**Predict:**

In [ ]:
print(Point.from_text("3, 4"))

`cls(int(x), int(y))` is `Point(3, 4)`. A **static method** gets neither the object nor the class: it's a plain function that lives in the class because it belongs there. `Point.origin_name()` returns `"the origin"`.

## Inheritance

In [ ]:
import math


class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        raise NotImplementedError

    def describe(self):
        return f"{self.name} with area {self.area():.2f}"


class Square(Shape):
    def __init__(self, side):
        super().__init__("square")
        self.side = side

    def area(self):
        return self.side**2


class Circle(Shape):
    def __init__(self, radius):
        super().__init__("circle")
        self.radius = radius

    def area(self):
        return math.pi * self.radius**2

`class Square(Shape)`: a `Square` **is a** `Shape`. It gets every method of `Shape` (it **inherits** them) and may replace some with its own (it **overrides** them). `super().__init__("square")` runs `Shape`'s `__init__` on this object.

**Predict:**

In [ ]:
print(Square(3).describe())

**Predict:**

In [ ]:
shapes = [Square(2), Circle(1), Square(1)]
for shape in shapes:
    print(shape.describe())

`describe` is only written once, in `Shape`. When it calls `self.area()`, Python looks up `area` on the actual object, so each shape uses its own. One loop, different behaviour: this is called **polymorphism**.

**Predict:**

In [ ]:
print(Shape("blob").describe())

**Predict:**

In [ ]:
print(isinstance(Square(1), Shape))

In [ ]:
print(isinstance(Square(1), Circle))

In [ ]:
print(type(Square(1)) is Shape)

**Predict:**

In [ ]:
class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return self.base * self.height / 2


print(Triangle(4, 3).describe())

`Triangle` has its own `__init__`, so `Shape.__init__` never runs and nobody sets `self.name`. If a subclass writes `__init__`, it usually calls `super().__init__(...)` first.

A subclass can also extend a method instead of replacing it. **Predict:**

In [ ]:
class LoudSquare(Square):
    def describe(self):
        return super().describe().upper() + "!"


print(LoudSquare(2).describe())

## A room

**Predict:**

In [ ]:
class Room:
    def __init__(self, width, length):
        self.width = width
        self.length = length

    def area(self):
        return self.width * self.length


def total_area(things):
    return sum(thing.area() for thing in things)


print(total_area([Square(2), Room(3, 4)]))

`Room` isn't a `Shape`, and it doesn't matter: `total_area` only calls `.area()`, and Python only checks, at that moment, that the object has such a method. "If it walks like a duck and quacks like a duck, it's a duck": **duck typing**. Other languages use **interfaces** to say "has these methods"; in Python, a base class whose methods raise `NotImplementedError` (like `Shape`) documents the same thing, and the `abc` module can enforce it.

## Exceptions are classes

In [ ]:
class NoInputError(ValueError):
    pass


def take_input(inputs):
    if len(inputs) == 0:
        raise NoInputError("no input left")
    return inputs.pop(0)

**Predict:**

In [ ]:
try:
    take_input([])
except ValueError as error:
    print("caught:", error)

The classes Python searches, in order:

In [ ]:
print(NoInputError.__mro__)

`NoInputError` is a `ValueError`, so `except ValueError` catches it (and so does `pytest.raises(ValueError)`). `__mro__` lists where Python looks for methods, in order: the class, its parent, and so on up to `object`, which every class inherits from. `pass` is a body that does nothing; the class only needs a name.

## Words you'll hear

Object-oriented programming (OOP) comes with its own vocabulary. You've seen all of it today:

- **Class** and **object** (or **instance**): `Account` is the class, the blueprint; `ada` and `bob` are objects made from it. **Attributes** are an object's data, **methods** the functions that work on it.
- **Encapsulation**: data and the methods that work on it live together, and outside code goes through the methods, as with `deposit` and `withdraw` guarding the balance. In Python by convention (`_balance`) and with properties; Java and C# enforce it with `private`.
- **Inheritance**: a class built on another one, taking over its methods (`Square(Shape)`). "Is a": a square is a shape.
- **Polymorphism**: one call, different behaviour depending on the object (`shape.area()` in the loop). In Python, duck typing gives you this even without a common base class.
- **Abstraction**: using something through what it does, without knowing how. `total_area` calls `.area()` and doesn't care how each shape computes it; you use `with open(...)` without knowing how files are closed.
- **Composition**: an object that holds other objects as attributes ("has a"), like a `Playlist` that has a list of songs.

These words come up in other courses, in job interviews, and in codebases at work, often with Java in mind. The ideas are the same in Python; the rules are looser.

## When to write a class

- When some state and the functions that change it belong together, and you may need several of them (several accounts, two computers). A class with only `__init__` and one other method is usually a function in disguise.
- Keep inheritance shallow. One level, as with `Shape`, covers most uses. To reuse code, an object that **has** another one (an attribute) is often clearer than one that **is** another one.
- Don't subclass `list` or `dict` to add a method: many of their methods return a plain `list` or `dict` again, not your class. If you need that, `collections.UserList` and `UserDict` exist.
- You'll meet more: **mixins**, small classes that only add methods, combined with multiple inheritance (`class A(B, C)`), are common in web frameworks. Read them; don't start with them.

# Part 2: Describing data

## Type annotations

In [ ]:
def area(width: float, height: float) -> float:
    return width * height

`width: float` says "`width` is meant to be a float"; `-> float` says what the function returns.

**Predict:**

In [ ]:
print(area(2, 3))

In [ ]:
print(area("ab", 3))

Python doesn't check annotations when it runs your code. They are for people and tools: VS Code shows them when you call the function, a type checker (such as `mypy`) can find mismatches without running the code, and FastAPI, later in the course, reads them to check the data in a web request. An `int` where a `float` is expected is fine.

Annotations for containers, and for "or":

In [ ]:
def find(grid: list[list[int]], value: int) -> tuple[int, int] | None:
    for row, line in enumerate(grid):
        for col, cell in enumerate(line):
            if cell == value:
                return (row, col)
    return None


scores: dict[str, int] = {"Ada": 3}
print(find.__annotations__)

`list[int]` is a list of ints, `dict[str, int]` a dict from strings to ints, `tuple[int, int]` a pair, and `X | None` means "an `X`, or `None`". Like `__name__` and `__doc__`, the annotations are stored on the function.

## Dataclasses

In [ ]:
from dataclasses import dataclass


@dataclass
class Point:
    x: int
    y: int

In [ ]:
p = Point(1, 2)
print(p)

In [ ]:
print(Point(1, 2) == Point(1, 2))

**Predict:**

In [ ]:
Point(1)

In [ ]:
print(Point("a", "b"))

`@dataclass` reads the annotated names in the class body, the **fields**, and writes `__init__`, `__repr__` and `__eq__` for you, much like Part 1's `Point`:

```python
def __init__(self, x: int, y: int):
    self.x = x
    self.y = y

def __repr__(self):
    return f"Point(x={self.x}, y={self.y})"

def __eq__(self, other):
    return (self.x, self.y) == (other.x, other.y)
```

The annotations still aren't checked: `Point("a", "b")` works. You can add methods as in any class:

In [ ]:
@dataclass
class Point:
    x: int
    y: int

    def distance_from_origin(self) -> int:
        return abs(self.x) + abs(self.y)


print(Point(3, -4).distance_from_origin())

## Defaults

In [ ]:
@dataclass
class Player:
    name: str
    score: int = 0


print(Player("Ada"))

A list as the default:

In [ ]:
@dataclass
class Team:
    name: str
    members: list[str] = []

Dataclasses refuse a mutable default: all teams would share one list (the trap from `Pet`). `field(default_factory=list)` says "call `list()` for every new object", so each one gets its own:

In [ ]:
from dataclasses import field


@dataclass
class Team:
    name: str
    members: list[str] = field(default_factory=list)

**Predict:**

In [ ]:
red = Team("red")
blue = Team("blue")
red.members.append("Ada")
print(blue.members)

Fields without a default come first: `name: str` after `members: ... = field(...)` would be a `TypeError`, for the same reason a function parameter without a default can't follow one with a default.

## Frozen dataclasses

In [ ]:
@dataclass(frozen=True)
class Position:
    row: int
    col: int

**Predict:**

In [ ]:
here = Position(1, 2)
here.row = 5

**Predict:**

In [ ]:
risks = {Position(0, 0): 1, Position(0, 1): 9}
print(risks[Position(0, 1)])

A frozen dataclass can't be changed after it's made, so it can have a hash, and it works as a dict key or in a set, like a tuple. Unlike `(0, 1)`, `Position(0, 1)` says which number is the row.

## Tuple, dict or dataclass?

| | `(1, 2)` | `{"row": 1, "col": 2}` | `Position(row=1, col=2)` |
|---|---|---|---|
| says what each part is | no | yes | yes |
| typo in a name (`col` vs `column`) | — | found when the line runs, or never | `AttributeError` / `TypeError` |
| methods | no | no | yes |
| as a dict key | yes | no | if frozen |
| shown by `print` | `(1, 2)` | `{'row': 1, 'col': 2}` | `Position(row=1, col=2)` |

A tuple is fine for something small and obvious, such as `(row, col)` inside one function. Once data travels through several functions, or has more than two or three parts, a dataclass documents itself.